# Copilot Org Data Loader

Ingests one or more **org data** CSVs (Entra/HRIS export with user → organisation/department mapping) into a flat Delta table consumed by the **AI-in-One Dashboard** and **AI Business Value Dashboard** Fabric variants.

**Inputs**: CSV(s) landed in `Files/org_raw/` of this Lakehouse. Accepted PersonId column variants: `User Principal Name`, `userPrincipalName`, `UserPrincipalName`, `UPN`, `PersonId`. The `Department` column (any case/spacing variant) is renamed to `Organization`. The `jobTitle` column is renamed to `JobTitle` for ABV-side calc-column compatibility.

**Output**: Lakehouse Delta table `dbo.copilot_org_data`. Delta tables disallow spaces / `,;{}()\n\t=` in column names, so the loader sanitises all column names to underscore-safe form after canonicalisation.

**Schedule**: run on the same cadence as your HRIS/Entra export (typically weekly).


## 1. Configuration

In [ ]:
RAW_PATH     = 'Files/org_raw/*.csv'      # raw HRIS/Entra org-data CSVs
OUTPUT_TABLE = 'copilot_org_data'         # Delta table consumed by the PBIT
WRITE_MODE   = 'overwrite'                # 'overwrite' for full snapshots

## 2. Read raw CSVs and trim column names
Trimming first matches the existing M-query behaviour — exports sometimes contain leading/trailing spaces in headers.

In [ ]:
import re
from pyspark.sql import functions as F

raw = (spark.read
    .option('header', 'true')
    .option('multiline', 'true')
    .option('escape', '"')
    .csv(RAW_PATH))

# Trim column names (matches M-query Table.TransformColumnNames step)
df = raw.toDF(*[c.strip() for c in raw.columns])

print(f'Raw rows: {df.count():,}')
print('Detected columns:', df.columns)

## 3. Detect column variants and canonicalise
PersonId and Department detection use a normalisation function (lowercase, strip punctuation/spaces) — same approach as the existing M-query.

In [ ]:
def normalize(s: str) -> str:
    return re.sub(r'[\s_\-.\(\)\[\]]', '', s).lower()

person_id_targets = {'userprincipalname', 'upn', 'personid'}
department_targets = {'department'}
job_title_targets  = {'jobtitle'}

col_person_id  = next((c for c in df.columns if normalize(c) in person_id_targets), None)
col_department = next((c for c in df.columns if normalize(c) in department_targets), None)
col_job_title  = next((c for c in df.columns if normalize(c) in job_title_targets and c != 'JobTitle'), None)

print(f'PersonId column detected:    {col_person_id}')
print(f'Department column detected:  {col_department}')
print(f'jobTitle column detected:    {col_job_title}')

if col_person_id is not None and col_person_id != 'PersonId':
    df = df.withColumnRenamed(col_person_id, 'PersonId')

if col_department is not None and col_department != 'Organization':
    df = df.withColumnRenamed(col_department, 'Organization')

if col_job_title is not None:
    df = df.withColumnRenamed(col_job_title, 'JobTitle')

## 4. Add `PersonId_Normalized` and `TotalEmployees`

In [ ]:
if 'PersonId' in df.columns:
    df = df.withColumn(
        'PersonId_Normalized',
        F.when(F.col('PersonId').isNull(), None)
         .otherwise(F.lower(F.trim(F.col('PersonId').cast('string'))))
    )

if 'TotalEmployees' not in df.columns:
    total = df.count()
    df = df.withColumn('TotalEmployees', F.lit(str(total)))

print(f'Final rows: {df.count():,}')

## 5. Sanitize column names for Delta
Delta rejects ` ,;{}()\n\t=` in column names — replace each invalid character with `_`.

In [ ]:
_INVALID = re.compile(r'[ ,;{}()\n\t=]')

def sanitize(name: str) -> str:
    return _INVALID.sub('_', name)

df = df.toDF(*[sanitize(c) for c in df.columns])
print('Sanitized columns:', df.columns)

## 6. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

print(f'Rows written to {OUTPUT_TABLE}: {spark.table(OUTPUT_TABLE).count():,}')

## 7. Verify
Spot-check `PersonId_Normalized` is populated and `Organization` looks like real department names.

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
cols_to_show = [c for c in ['PersonId', 'PersonId_Normalized', 'Organization', 'JobTitle'] if c in tbl.columns]
tbl.select(*cols_to_show).show(10, truncate=False)

if 'Organization' in tbl.columns:
    tbl.groupBy('Organization').count().orderBy(F.desc('count')).show(20, truncate=False)

---
**Connect the PBIT**: this table is consumed by the `Chat + Agent Org Data` query in the AI-in-One and AI Business Value dashboards. Once this notebook has run, you can leave the `Org Data File` parameter blank — refresh will source from `dbo.copilot_org_data` directly via the Fabric SQL endpoint.